# 🔢 Ordinal Encoding

When working in Machine Learning, some categories have a **natural order**.

Examples:

| Feature | Values |
|---------|--------|
| Education | Primary → Secondary → University |
| Size | Small → Medium → Large |
| Rating | Poor → Average → Good → Excellent |

These are called **Ordinal Categories**.

For these, we use:

> ## ✅ Ordinal Encoding

---

## 💡 Core Idea

Assign numbers that **respect the natural order** of categories.

| Size | Ordinal Encoded |
|------|----------------|
| Small | 1 |
| Medium | 2 |
| Large | 3 |

The number reflects the **rank**:
- Small is least → gets lowest number
- Large is most → gets highest number

Order is **preserved and meaningful**.

---

## 📌 Basic Example

Suppose:

| Education |
|-----------|
| Primary |
| University |
| Secondary |
| Primary |

After Ordinal Encoding:

| Education | Encoded |
|-----------|---------|
| Primary | 1 |
| University | 3 |
| Secondary | 2 |
| Primary | 1 |

Model now correctly understands:

```
University (3) > Secondary (2) > Primary (1)
```

---

## 🔄 Difference From Label Encoding

Both assign numbers to categories.
But there is one critical difference:

| Feature | Label Encoding | Ordinal Encoding |
|---------|---------------|-----------------|
| Order respected? | ❌ No (alphabetical/random) | ✅ Yes (you define the order) |
| You control ranking? | ❌ No | ✅ Yes |
| Best for? | Nominal or tree models | Ordinal categories only |

### Label Encoding (wrong for ordinal):

| Size | Label Encoded (alphabetical) |
|------|------------------------------|
| Large | 0 |
| Medium | 1 |
| Small | 2 |

Large gets 0 → Model thinks Large is smallest. ❌ Wrong!

### Ordinal Encoding (correct):

| Size | Ordinal Encoded |
|------|----------------|
| Small | 1 |
| Medium | 2 |
| Large | 3 |

Small gets 1 → Model correctly knows Small < Medium < Large. ✅

---

## 🧪 Dry Run — Step By Step

Suppose dataset:

| Rating |
|--------|
| Good |
| Poor |
| Excellent |
| Average |
| Good |

### Step 1: Define The Order

You must **manually define** the correct order:

```
Poor < Average < Good < Excellent
```

### Step 2: Assign Numbers

| Rating | Encoded |
|--------|---------|
| Poor | 1 |
| Average | 2 |
| Good | 3 |
| Excellent | 4 |

### Step 3: Replace Original Values

| Rating | Encoded |
|--------|---------|
| Good | 3 |
| Poor | 1 |
| Excellent | 4 |
| Average | 2 |
| Good | 3 |

### ✅ Final Encoded Dataset

| Rating | Encoded |
|--------|---------|
| Good | 3 |
| Poor | 1 |
| Excellent | 4 |
| Average | 2 |
| Good | 3 |

---

## 🐍 Python Code

### Method 1: Using scikit-learn OrdinalEncoder

```python
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder

df = pd.DataFrame({
    'Size': ['Small', 'Large', 'Medium', 'Small', 'Large']
})

# Define the correct order manually
encoder = OrdinalEncoder(categories=[['Small', 'Medium', 'Large']])

df['Size_Encoded'] = encoder.fit_transform(df[['Size']])

print(df)
```

Output:

| Size | Size\_Encoded |
|------|--------------|
| Small | 0.0 |
| Large | 2.0 |
| Medium | 1.0 |
| Small | 0.0 |
| Large | 2.0 |

> ⚠️ Note: scikit-learn starts from **0** by default.

---

### Method 2: Using pandas map() — Most Control

```python
import pandas as pd

df = pd.DataFrame({
    'Rating': ['Good', 'Poor', 'Excellent', 'Average', 'Good']
})

# Define custom mapping
rating_order = {
    'Poor'      : 1,
    'Average'   : 2,
    'Good'      : 3,
    'Excellent' : 4
}

df['Rating_Encoded'] = df['Rating'].map(rating_order)

print(df)
```

Output:

| Rating | Rating\_Encoded |
|--------|----------------|
| Good | 3 |
| Poor | 1 |
| Excellent | 4 |
| Average | 2 |
| Good | 3 |

> ✅ `map()` gives you **full control** over the order.

---

### Method 3: Using pandas replace()

```python
import pandas as pd

df = pd.DataFrame({
    'Education': ['Primary', 'University', 'Secondary', 'Primary']
})

df['Education_Encoded'] = df['Education'].replace({
    'Primary'    : 1,
    'Secondary'  : 2,
    'University' : 3
})

print(df)
```

Output:

| Education | Education\_Encoded |
|-----------|------------------|
| Primary | 1 |
| University | 3 |
| Secondary | 2 |
| Primary | 1 |

---

### Method 4: Multiple Ordinal Columns At Once

```python
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder

df = pd.DataFrame({
    'Size'      : ['Small', 'Large', 'Medium', 'Small'],
    'Education' : ['Primary', 'University', 'Secondary', 'University']
})

encoder = OrdinalEncoder(categories=[
    ['Small', 'Medium', 'Large'],         # order for Size
    ['Primary', 'Secondary', 'University'] # order for Education
])

df[['Size_Enc', 'Education_Enc']] = encoder.fit_transform(
    df[['Size', 'Education']]
)

print(df)
```

Output:

| Size | Education | Size\_Enc | Education\_Enc |
|------|-----------|----------|---------------|
| Small | Primary | 0.0 | 0.0 |
| Large | University | 2.0 | 2.0 |
| Medium | Secondary | 1.0 | 1.0 |
| Small | University | 0.0 | 2.0 |

---

## ⚠️ Handling Unknown Categories

What if a new value appears that was not in training data?

```python
from sklearn.preprocessing import OrdinalEncoder
import pandas as pd

encoder = OrdinalEncoder(
    categories=[['Small', 'Medium', 'Large']],
    handle_unknown='use_encoded_value',
    unknown_value=-1   # unknown gets -1
)

df_train = pd.DataFrame({'Size': ['Small', 'Medium', 'Large']})
df_test  = pd.DataFrame({'Size': ['Small', 'XLarge']})  # XLarge is unknown

encoder.fit(df_train[['Size']])

print(encoder.transform(df_test[['Size']]))
# Output: [[ 0.]
#          [-1.]]   ← XLarge becomes -1
```

---

## ✅ When To Use Ordinal Encoding

### 1. Ordinal Categories

Categories that have a **clear natural ranking**:

| Feature | Order |
|---------|-------|
| Education | Primary < Secondary < University |
| Size | XS < S < M < L < XL |
| Satisfaction | Terrible < Bad < OK < Good < Excellent |
| Priority | Low < Medium < High < Critical |

✅ **Perfect use case.**

### 2. Tree-Based Models

Models like:
- Decision Tree
- Random Forest
- XGBoost
- LightGBM

These split by threshold and **handle ordinal numbers well**.

---

## ❌ When NOT To Use Ordinal Encoding

### Nominal Categories (No Order)

| City | Ordinal Encoded |
|------|----------------|
| Lahore | 1 |
| Karachi | 2 |
| Islamabad | 3 |

Model will think Islamabad > Karachi > Lahore. ❌ Wrong!

> Use **One Hot Encoding** for nominal categories.

### Linear Models With Nominal Data

Models like Linear Regression will treat the numbers as distances.
This creates fake mathematical relationships.

---

## ⚖️ All Three Encodings Compared

| Feature | Label Encoding | Ordinal Encoding | One Hot Encoding |
|---------|---------------|-----------------|-----------------|
| Best for | Nominal + tree models | Ordinal categories | Nominal categories |
| Preserves order | ❌ Random | ✅ You define it | ❌ No order needed |
| Extra columns | ❌ None | ❌ None | ✅ One per category |
| Memory usage | ✅ Low | ✅ Low | ❌ Higher |
| Linear models safe | ❌ No | ✅ Yes (if ordinal) | ✅ Yes |
| Tree models safe | ✅ Yes | ✅ Yes | ✅ Yes |

---

## 👍 Advantages

| # | Advantage |
|---|-----------|
| 1 | **Preserves natural order** — meaningful numbers |
| 2 | **No extra columns** — memory efficient |
| 3 | **You control the mapping** — full flexibility |
| 4 | **Works well with all model types** for ordinal data |
| 5 | **Simple and fast** to implement |

---

## 👎 Disadvantages

| # | Disadvantage |
|---|-------------|
| 1 | **You must manually define the order** — easy to make mistakes |
| 2 | **Wrong if applied to nominal data** — misleads models |
| 3 | **Assumes equal distance** between ranks (1→2→3 are equally spaced) |

---

## 🧠 Key Rule To Remember

> **Always ask: Does this category have a natural order?**

```
Yes → Ordinal Encoding ✅
No  → One Hot Encoding ✅
```

| Category Type | Example | Best Encoding |
|--------------|---------|--------------|
| Has order (ordinal) | Size, Education, Rating | Ordinal Encoding |
| No order (nominal) | Colors, Cities, Names | One Hot Encoding |
| Tree model + nominal | Any category | Label Encoding (safe) |

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, LabelEncoder

In [3]:
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder

# -------------------------
# DATA
# -------------------------
df = pd.DataFrame({
    "Size": ["Small", "Medium", "Large", "Medium", "Small"]
})

print("ORIGINAL DATA:")
print(df)

# -------------------------
# DEFINE ORDER (VERY IMPORTANT)
# -------------------------
size_order = [["Small", "Medium", "Large"]]

# -------------------------
# CREATE ENCODER
# -------------------------
encoder = OrdinalEncoder(categories=size_order)

# -------------------------
# FIT + TRANSFORM
# -------------------------
df["Size_encoded"] = encoder.fit_transform(df[["Size"]])

# -------------------------
# RESULT
# -------------------------
print("\nENCODED DATA:")
print(df)

ORIGINAL DATA:
     Size
0   Small
1  Medium
2   Large
3  Medium
4   Small

ENCODED DATA:
     Size  Size_encoded
0   Small           0.0
1  Medium           1.0
2   Large           2.0
3  Medium           1.0
4   Small           0.0


# Multiple Columns

`Ulike Label encoder it can Detect Multiple cols at once`

LabelEncoder
- Designed for target labels (y)
- Only understands one column at a time
- Input must be 1D

OrdinalEncoder
- Designed for feature columns (X)
- Understands multiple columns together
- Input is 2D matrix

> OrdinalEncoder().fit_transform(df[["col1", "col2", "col3"]])

In [5]:
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder

# =========================================================
# 🧠 STEP 1: RAW DATA
# =========================================================
# This dataset contains ordered categorical variables
# ---------------------------------------------------------

df = pd.DataFrame({
    "Size": ["Small", "Medium", "Large", "Medium", "Small", "Large"],
    "Education": ["Bachelor", "HighSchool", "PhD", "Master", "Bachelor", "Master"],
    "Rating": ["Good", "Excellent", "Average", "Good", "Poor", "Excellent"]
})

print("ORIGINAL DATA:\n", df)


# =========================================================
# 🧠 STEP 2: DEFINE ORDER (MOST IMPORTANT PART)
# =========================================================
# WE manually tell sklearn the correct ranking of categories
# ---------------------------------------------------------

size_order = ["Small", "Medium", "Large"]

education_order = ["HighSchool", "Bachelor", "Master", "PhD"]

rating_order = ["Poor", "Average", "Good", "Excellent"]

# Combine all column orders
categories = [size_order, education_order, rating_order]


# =========================================================
# 🧠 STEP 3: CREATE ORDINAL ENCODER
# =========================================================
# This encoder respects OUR defined order
# ---------------------------------------------------------

encoder = OrdinalEncoder(categories=categories)


# =========================================================
# 🧠 STEP 4: FIT + TRANSFORM
# =========================================================
# FIT → learns mapping from our defined order
# TRANSFORM → converts categories into numbers
# ---------------------------------------------------------

encoded = encoder.fit_transform(df)


# =========================================================
# 🧠 STEP 5: CONVERT BACK TO DATAFRAME
# =========================================================

encoded_df = pd.DataFrame(encoded, columns=df.columns)

print("\nENCODED DATA:\n", encoded_df)


# =========================================================
# 🧠 STEP 6: SEE INTERNAL MAPPING
# =========================================================
print("\nMAPPING LEARNED:")
print("Size:", size_order)
print("Education:", education_order)
print("Rating:", rating_order)

ORIGINAL DATA:
      Size   Education     Rating
0   Small    Bachelor       Good
1  Medium  HighSchool  Excellent
2   Large         PhD    Average
3  Medium      Master       Good
4   Small    Bachelor       Poor
5   Large      Master  Excellent

ENCODED DATA:
    Size  Education  Rating
0   0.0        1.0     2.0
1   1.0        0.0     3.0
2   2.0        3.0     1.0
3   1.0        2.0     2.0
4   0.0        1.0     0.0
5   2.0        2.0     3.0

MAPPING LEARNED:
Size: ['Small', 'Medium', 'Large']
Education: ['HighSchool', 'Bachelor', 'Master', 'PhD']
Rating: ['Poor', 'Average', 'Good', 'Excellent']


# ML Pipeline


In [ ]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, LabelEncoder

# =========================================================
# 🧠 STEP 1: RAW DATA (REAL-WORLD STUDENT SELECTION DATA)
# =========================================================
# We intentionally mix:
# - nominal features (no order)
# - ordinal features (have order)
# - target variable
# ---------------------------------------------------------

df = pd.DataFrame({
    "City": ["Lahore", "Karachi", "Lahore", "Islamabad", "Multan", "Karachi"],
    "Gender": ["Male", "Female", "Female", "Male", "Male", "Female"],
    "Education": ["Bachelor", "HighSchool", "PhD", "Master", "Bachelor", "Master"],
    "Experience_Level": ["Low", "Medium", "High", "Medium", "Low", "High"],
    "Selected": ["No", "Yes", "Yes", "No", "Yes", "No"]
})

print("ORIGINAL DATA:\n", df)


# =========================================================
# 🧠 STEP 2: SPLIT INTO TRAIN / TEST
# =========================================================
# TRAIN → model learns patterns
# TEST → model evaluates unseen data
# ---------------------------------------------------------

train = df.iloc[:4]

test = pd.DataFrame({
    "City": ["Lahore", "NewCity"],         # unseen ❌
    "Gender": ["Male", "Female"],
    "Education": ["Bachelor", "Diploma"],  # unseen ❌
    "Experience_Level": ["Low", "Expert"], # unseen ❌
    "Selected": ["No", "Yes"]
})

print("\nTRAIN DATA:\n", train)
print("\nTEST DATA:\n", test)


# =========================================================
# 🧠 STEP 3: SPLIT FEATURES (X) AND TARGET (y)
# =========================================================

X_train = train.drop("Selected", axis=1)
X_test = test.drop("Selected", axis=1)

y_train = train["Selected"]
y_test = test["Selected"]


# =========================================================
# 🧠 STEP 4: LABEL ENCODING (TARGET y)
# =========================================================
# WHY?
# ML models understand only numbers
# Yes/No → 1/0
# ---------------------------------------------------------

label_encoder = LabelEncoder()

y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

print("\nTARGET CLASSES:", label_encoder.classes_)
print("Encoded y_train:", y_train_encoded)


# =========================================================
# 🧠 STEP 5: ONE HOT ENCODING (Nominal Features)
# =========================================================
# These have NO ORDER:
# - City
# - Gender
# ---------------------------------------------------------

onehot_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

X_train_onehot = onehot_encoder.fit_transform(X_train[["City", "Gender"]])
X_test_onehot = onehot_encoder.transform(X_test[["City", "Gender"]])


# =========================================================
# 🧠 STEP 6: ORDINAL ENCODING (Ordered Features)
# =========================================================
# These HAVE REAL ORDER:
# - Education
# - Experience Level
# ---------------------------------------------------------

education_order = ["HighSchool", "Bachelor", "Master", "PhD"]
experience_order = ["Low", "Medium", "High"]

ordinal_encoder = OrdinalEncoder(
    categories=[education_order, experience_order],
    handle_unknown="use_encoded_value",
    unknown_value=-1  # Diploma is unknown Value so it will be "-1"
)

X_train_ordinal = ordinal_encoder.fit_transform(X_train[["Education", "Experience_Level"]])
X_test_ordinal = ordinal_encoder.transform(X_test[["Education", "Experience_Level"]])


# =========================================================
# 🧠 STEP 7: COMBINE ALL FEATURES (FINAL ML INPUT)
# =========================================================

import numpy as np

X_train_final = np.hstack([X_train_onehot, X_train_ordinal])
X_test_final = np.hstack([X_test_onehot, X_test_ordinal])


# =========================================================
# 🧠 STEP 8: CONVERT TO DATAFRAMES (FOR VISIBILITY)
# =========================================================

train_df = pd.DataFrame(X_train_final)
test_df = pd.DataFrame(X_test_final)

print("\nFINAL TRAIN FEATURES:\n", train_df)
print("\nFINAL TEST FEATURES:\n", test_df)


# =========================================================
# 🧠 STEP 9: FINAL FLOW EXPLANATION
# =========================================================
"""
FULL PIPELINE FLOW:

1. RAW DATA
   → mix of nominal + ordinal + target

2. SPLIT DATA
   → train = learning phase
   → test = unseen evaluation

3. X / y SPLIT
   → X = features
   → y = target

4. LABEL ENCODER (y)
   → Yes/No → 1/0

5. ONE HOT ENCODER (nominal features)
   → City, Gender → binary vectors
   → no fake ordering

6. ORDINAL ENCODER (ordered features)
   → Education, Experience
   → uses real-world ranking

7. COMBINE FEATURES
   → merge both encodings into final ML matrix

8. TEST TRANSFORM
   → unseen categories → ignored / zeros

IMPORTANT RULE:
- fit ONLY on train
- transform ONLY on test
- NEVER leak test data
"""

ORIGINAL DATA:
         City  Gender   Education Experience_Level Selected
0     Lahore    Male    Bachelor              Low       No
1    Karachi  Female  HighSchool           Medium      Yes
2     Lahore  Female         PhD             High      Yes
3  Islamabad    Male      Master           Medium       No
4     Multan    Male    Bachelor              Low      Yes
5    Karachi  Female      Master             High       No

TRAIN DATA:
         City  Gender   Education Experience_Level Selected
0     Lahore    Male    Bachelor              Low       No
1    Karachi  Female  HighSchool           Medium      Yes
2     Lahore  Female         PhD             High      Yes
3  Islamabad    Male      Master           Medium       No

TEST DATA:
       City  Gender Education Experience_Level Selected
0   Lahore    Male  Bachelor              Low       No
1  NewCity  Female   Diploma           Expert      Yes

TARGET CLASSES: ['No' 'Yes']
Encoded y_train: [0 1 1 0]

FINAL TRAIN FEATURES:
    

'\nFULL PIPELINE FLOW:\n\n1. RAW DATA\n   → mix of nominal + ordinal + target\n\n2. SPLIT DATA\n   → train = learning phase\n   → test = unseen evaluation\n\n3. X / y SPLIT\n   → X = features\n   → y = target\n\n4. LABEL ENCODER (y)\n   → Yes/No → 1/0\n\n5. ONE HOT ENCODER (nominal features)\n   → City, Gender → binary vectors\n   → no fake ordering\n\n6. ORDINAL ENCODER (ordered features)\n   → Education, Experience\n   → uses real-world ranking\n\n7. COMBINE FEATURES\n   → merge both encodings into final ML matrix\n\n8. TEST TRANSFORM\n   → unseen categories → ignored / zeros\n\nIMPORTANT RULE:\n- fit ONLY on train\n- transform ONLY on test\n- NEVER leak test data\n'

In [1]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, LabelEncoder
import numpy as np

# =========================================================
# 🧠 STEP 1: RAW DATA (REALISTIC HR DATASET)
# =========================================================

df = pd.DataFrame({
    "Education": ["HighSchool", "Bachelor", "Master", "PhD", "Bachelor", "Master"],
    "Experience": ["Fresher", "Junior", "Senior", "Senior", "Junior", "Fresher"],
    "JobRole": ["Developer", "Designer", "Manager", "Developer", "Manager", "Designer"],
    "City": ["Lahore", "Karachi", "Lahore", "Islamabad", "Multan", "Karachi"],
    "Shortlisted": ["No", "Yes", "Yes", "No", "Yes", "No"]
})

print("ORIGINAL DATA:\n", df)


# =========================================================
# 🧠 STEP 2: TRAIN / TEST SPLIT (SIMULATING REAL WORLD)
# =========================================================

train = df.iloc[:4]

test = pd.DataFrame({
    "Education": ["Bachelor", "Diploma"],     # unseen ❌
    "Experience": ["Junior", "Expert"],       # unseen ❌
    "JobRole": ["Developer", "Architect"],    # unseen ❌
    "City": ["Lahore", "NewCity"],            # unseen ❌
    "Shortlisted": ["No", "Yes"]
})

print("\nTRAIN DATA:\n", train)
print("\nTEST DATA:\n", test)


# =========================================================
# 🧠 STEP 3: SPLIT X AND y
# =========================================================

X_train = train.drop("Shortlisted", axis=1)
X_test = test.drop("Shortlisted", axis=1)

y_train = train["Shortlisted"]
y_test = test["Shortlisted"]


# =========================================================
# 🧠 STEP 4: LABEL ENCODING (TARGET y)
# =========================================================

label_encoder = LabelEncoder()

y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

print("\nTARGET CLASSES:", label_encoder.classes_)
print("Encoded y_train:", y_train_encoded)


# =========================================================
# 🧠 STEP 5: ORDINAL ENCODING (ORDERED FEATURES)
# =========================================================

education_order = ["HighSchool", "Bachelor", "Master", "PhD"]

experience_order = ["Fresher", "Junior", "Senior"]

ordinal_encoder = OrdinalEncoder(
    categories=[education_order, experience_order],
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

X_train_ordinal = ordinal_encoder.fit_transform(
    X_train[["Education", "Experience"]]
)

X_test_ordinal = ordinal_encoder.transform(
    X_test[["Education", "Experience"]]
)


# =========================================================
# 🧠 STEP 6: ONE HOT ENCODING (UNORDERED FEATURES)
# =========================================================

onehot_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

X_train_onehot = onehot_encoder.fit_transform(
    X_train[["JobRole", "City"]]
)

X_test_onehot = onehot_encoder.transform(
    X_test[["JobRole", "City"]]
)


# =========================================================
# 🧠 STEP 7: COMBINE ALL FEATURES
# =========================================================

X_train_final = np.hstack([X_train_ordinal, X_train_onehot])
X_test_final = np.hstack([X_test_ordinal, X_test_onehot])


# =========================================================
# 🧠 STEP 8: CONVERT TO DATAFRAMES
# =========================================================

train_df = pd.DataFrame(X_train_final)
test_df = pd.DataFrame(X_test_final)

print("\nFINAL TRAIN FEATURES:\n", train_df)
print("\nFINAL TEST FEATURES:\n", test_df)


# =========================================================
# 🧠 STEP 9: FLOW SUMMARY
# =========================================================
"""
FULL FLOW:

1. RAW DATA
   - Education (ordinal)
   - Experience (ordinal)
   - JobRole (nominal)
   - City (nominal)
   - Target (Shortlisted)

2. SPLIT
   - Train = learn patterns
   - Test = unseen evaluation

3. y ENCODING
   - Yes/No → 1/0

4. ORDINAL ENCODING
   - Education → ordered ranking
   - Experience → ordered ranking
   - unknown → -1

5. ONE HOT ENCODING
   - JobRole, City → binary vectors
   - unknown → ignored safely

6. COMBINE FEATURES
   - merge ordinal + onehot

7. FINAL ML MATRIX READY
"""

ORIGINAL DATA:
     Education Experience    JobRole       City Shortlisted
0  HighSchool    Fresher  Developer     Lahore          No
1    Bachelor     Junior   Designer    Karachi         Yes
2      Master     Senior    Manager     Lahore         Yes
3         PhD     Senior  Developer  Islamabad          No
4    Bachelor     Junior    Manager     Multan         Yes
5      Master    Fresher   Designer    Karachi          No

TRAIN DATA:
     Education Experience    JobRole       City Shortlisted
0  HighSchool    Fresher  Developer     Lahore          No
1    Bachelor     Junior   Designer    Karachi         Yes
2      Master     Senior    Manager     Lahore         Yes
3         PhD     Senior  Developer  Islamabad          No

TEST DATA:
   Education Experience    JobRole     City Shortlisted
0  Bachelor     Junior  Developer   Lahore          No
1   Diploma     Expert  Architect  NewCity         Yes

TARGET CLASSES: ['No' 'Yes']
Encoded y_train: [0 1 1 0]

FINAL TRAIN FEATURES:
    

'\nFULL FLOW:\n\n1. RAW DATA\n   - Education (ordinal)\n   - Experience (ordinal)\n   - JobRole (nominal)\n   - City (nominal)\n   - Target (Shortlisted)\n\n2. SPLIT\n   - Train = learn patterns\n   - Test = unseen evaluation\n\n3. y ENCODING\n   - Yes/No → 1/0\n\n4. ORDINAL ENCODING\n   - Education → ordered ranking\n   - Experience → ordered ranking\n   - unknown → -1\n\n5. ONE HOT ENCODING\n   - JobRole, City → binary vectors\n   - unknown → ignored safely\n\n6. COMBINE FEATURES\n   - merge ordinal + onehot\n\n7. FINAL ML MATRIX READY\n'